# Similarity-Only Matrix Viewer

This notebook computes and visualizes the cross-database atomic energy-level similarity matrix without applying threshold filtering or Hungarian assignment. It supports selection of the isoelectronic sequence, ion, database pair, level-index ranges, and figure dimensions, and saves publication-ready PDF figures to `map_data/photos/`.

Usage:
1. Run all cells.
2. Select `SEQ`, `ION`, `DB1`, and `DB2` in the widgets.
3. Optionally specify the level-index ranges and figure dimensions.
4. Click `Draw and save PDF`.

In [ ]:
import os
import re
import sys
import warnings
from contextlib import contextmanager
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

try:
    from pandas.errors import PerformanceWarning
except Exception:
    PerformanceWarning = Warning
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

import ipywidgets as widgets
from IPython.display import display, clear_output

# Robustly find cfadb project root.
# Works when cwd is the project root or the viewer folder.
cwd = Path.cwd().resolve()

if (cwd / "viewer" / "threshold_viewer.py").exists():
    PROJECT_ROOT = cwd
elif (cwd / "threshold_viewer.py").exists() and cwd.name == "viewer":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = None
    for p in [cwd] + list(cwd.parents):
        if (p / "viewer" / "threshold_viewer.py").exists():
            PROJECT_ROOT = p
            break
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            f"Cannot find project root from cwd={cwd}. "
            "Expected viewer/threshold_viewer.py under project root."
        )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from viewer import threshold_viewer as tv

CLEAN_DATA_DIR = PROJECT_ROOT / "clean_data"
THR_VIEW_ROOT = PROJECT_ROOT / "map_data" / "thr_view"
CFG_MODEL_ROOT = PROJECT_ROOT / "map_data" / "cfg_ts"
PHOTO_DIR = PROJECT_ROOT / "map_data" / "photos"
PHOTO_DIR.mkdir(parents=True, exist_ok=True)

# Similarity setting: same default as thr_viewer.ipynb.
EDGE_ABLATION = "xcos"
XCOS_KEEP = "cfg,phys,energy"
MODEL_PATH = None

# Same colors as viewer/threshold_viewer.py custom_colors.
CUSTOM_COLORS = [
    "#3A4750",
    "#048998",
    "#3bb4c1",
    "#9CDEC9",
    "#D6FFF2",
]


@contextmanager
def _suppress_plot_warnings():
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r".*DataFrame is highly fragmented.*",
            category=PerformanceWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message=r".*Creating a tensor from a list of numpy\.ndarrays is extremely slow.*",
            category=UserWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message=r".*DataFrame concatenation with empty or all-NA entries is deprecated.*",
            category=FutureWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message=r".*pd\.concat.*",
            category=FutureWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message=r".*merged_data.*",
            category=FutureWarning,
        )
        yield

print("cwd =", os.getcwd())
print("project root =", PROJECT_ROOT)
print("threshold_viewer =", Path(tv.__file__).resolve())
print("photo dir =", PHOTO_DIR)


In [ ]:
# -----------------------------
# Discovery helpers and defaults
# -----------------------------
DEFAULT_SEQ = "be-like"
DEFAULT_ION = "042623"
DEFAULT_DB1 = "atomdb313"
DEFAULT_DB2 = "spex30801"  # or "chianti1102"


def _prefer(options, preferred):
    options = list(options)
    if not options:
        return None
    return preferred if preferred in options else options[0]


def discover_seqs():
    if CFG_MODEL_ROOT.exists():
        seqs = sorted([p.name for p in CFG_MODEL_ROOT.iterdir() if p.is_dir()])
        if seqs:
            return seqs
    return [DEFAULT_SEQ]


def discover_ions():
    if CLEAN_DATA_DIR.exists():
        ions = sorted([p.stem for p in CLEAN_DATA_DIR.glob("*.h5")])
        if ions:
            return ions
    return [DEFAULT_ION]


def discover_dbs(ion):
    h5_path = CLEAN_DATA_DIR / f"{ion}.h5"
    if not h5_path.exists():
        return [DEFAULT_DB1, "chianti1102", DEFAULT_DB2]

    dbs = []
    with h5py.File(h5_path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                leaf = name.split("/")[-1]
                if leaf.startswith("lev_") and not leaf.endswith("_delete"):
                    dbs.append(leaf.replace("lev_", "", 1))
        f.visititems(visit)
    return sorted(set(dbs)) or [DEFAULT_DB1, "chianti1102", DEFAULT_DB2]


seq_options = discover_seqs()
ion_options = discover_ions()
db_options = discover_dbs(_prefer(ion_options, DEFAULT_ION))


In [ ]:
# -----------------------------
# Selectors
# -----------------------------
seq_w = widgets.Dropdown(
    options=seq_options,
    value=_prefer(seq_options, DEFAULT_SEQ),
    description="SEQ",
    layout=widgets.Layout(width="260px"),
)

ion_w = widgets.Dropdown(
    options=ion_options,
    value=_prefer(ion_options, DEFAULT_ION),
    description="ION",
    layout=widgets.Layout(width="260px"),
)

db1_w = widgets.Dropdown(
    options=db_options,
    value=_prefer(db_options, DEFAULT_DB1),
    description="DB1",
    layout=widgets.Layout(width="300px"),
)

db2_w = widgets.Dropdown(
    options=db_options,
    value=_prefer(db_options, DEFAULT_DB2),
    description="DB2",
    layout=widgets.Layout(width="300px"),
)

# Blank means full range. Values are actual level idx values when an idx column exists.
db1_min_w = widgets.Text(value="", description="DB1 idx min", placeholder="blank = all", layout=widgets.Layout(width="260px"))
db1_max_w = widgets.Text(value="", description="DB1 idx max", placeholder="blank = all", layout=widgets.Layout(width="260px"))
db2_min_w = widgets.Text(value="", description="DB2 idx min", placeholder="blank = all", layout=widgets.Layout(width="260px"))
db2_max_w = widgets.Text(value="", description="DB2 idx max", placeholder="blank = all", layout=widgets.Layout(width="260px"))

fig_w_w = widgets.FloatText(value=10.0, description="fig width", layout=widgets.Layout(width="220px"))
fig_h_w = widgets.FloatText(value=10.0, description="fig height", layout=widgets.Layout(width="220px"))
max_tick_w = widgets.IntSlider(value=30, min=5, max=120, step=5, description="max ticks", layout=widgets.Layout(width="420px"))

draw_btn = widgets.Button(description="Draw and save PDF", button_style="success", layout=widgets.Layout(width="180px"))
out = widgets.Output()


def _update_db_options(change=None):
    old_db1 = db1_w.value
    old_db2 = db2_w.value
    opts = discover_dbs(ion_w.value)
    db1_w.options = opts
    db2_w.options = opts
    db1_w.value = old_db1 if old_db1 in opts else _prefer(opts, DEFAULT_DB1)
    db2_w.value = old_db2 if old_db2 in opts else _prefer(opts, DEFAULT_DB2)


ion_w.observe(_update_db_options, names="value")
_update_db_options()

control_box = widgets.VBox([
    widgets.HBox([seq_w, ion_w]),
    widgets.HBox([db1_w, db2_w]),
    widgets.HBox([db1_min_w, db1_max_w]),
    widgets.HBox([db2_min_w, db2_max_w]),
    widgets.HBox([fig_w_w, fig_h_w]),
    max_tick_w,
    draw_btn,
    out,
])

display(control_box)


In [ ]:
# -----------------------------
# Similarity-only plotting
# -----------------------------
def _parse_optional_int(text):
    text = str(text).strip()
    if text == "":
        return None
    return int(float(text))


def _idx_values(info_df):
    if "idx" in info_df.columns:
        vals = pd.to_numeric(info_df["idx"], errors="coerce").to_numpy(dtype=float)
    else:
        vals = np.arange(len(info_df), dtype=float)
    return vals


def _select_rows_by_idx(info_df, idx_min=None, idx_max=None):
    vals = _idx_values(info_df)
    mask = np.isfinite(vals)
    if idx_min is not None:
        mask &= vals >= int(idx_min)
    if idx_max is not None:
        mask &= vals <= int(idx_max)
    rows = np.where(mask)[0].astype(int)
    if len(rows) == 0:
        raise ValueError(f"No rows found for idx range [{idx_min}, {idx_max}].")
    return rows


def _safe_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_")


def _format_range_for_name(vmin, vmax):
    left = "all" if vmin is None else str(vmin)
    right = "all" if vmax is None else str(vmax)
    return f"{left}-{right}"


def _same_threshold_viewer_norm(arr):
    # Use exactly the custom color list from viewer/threshold_viewer.py,
    # do not use matplotlib default colormaps here.
    cmap = ListedColormap(CUSTOM_COLORS)
    n_bins = len(CUSTOM_COLORS)

    vals = np.asarray(arr, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        raise ValueError("The selected similarity matrix contains no finite values.")

    vmin = float(vals.min())
    vmax = float(vals.max())
    if np.isclose(vmin, vmax):
        vmax = vmin + 1e-6
    bounds = np.linspace(vmin, vmax, n_bins + 1)
    norm = BoundaryNorm(bounds, cmap.N, clip=True)
    return cmap, norm, bounds


def plot_similarity_only_pdf(
    *,
    seq,
    ion,
    db1,
    db2,
    db1_idx_min=None,
    db1_idx_max=None,
    db2_idx_min=None,
    db2_idx_max=None,
    figsize=(10.0, 10.0),
    max_tick_labels=30,
):
    cfg_model_path = CFG_MODEL_ROOT / seq / "cfg_ts_model.pth"
    if not cfg_model_path.exists():
        raise FileNotFoundError(f"Cannot find cfg model: {cfg_model_path}")

    with _suppress_plot_warnings():
        pair_data = tv.load_pair_from_h5(
            ion=ion,
            db1=db1,
            db2=db2,
            clean_data_dir=CLEAN_DATA_DIR,
            seq=seq,
            cfg_model_path=cfg_model_path,
            thr_view_root=THR_VIEW_ROOT,
        )

        sim, sim_summary = tv.compute_similarity_for_pair(
            pair_data,
            model_path=MODEL_PATH,
            edge_ablation=EDGE_ABLATION,
            xcos_keep=XCOS_KEEP,
            summarize=True,
        )

    info1 = pair_data["info1"]
    info2 = pair_data["info2"]
    rows = _select_rows_by_idx(info1, db1_idx_min, db1_idx_max)
    cols = _select_rows_by_idx(info2, db2_idx_min, db2_idx_max)
    sim_sub = sim[np.ix_(rows, cols)]

    fig, ax = plt.subplots(figsize=figsize)
    cmap10, norm, bounds = _same_threshold_viewer_norm(sim_sub)
    im = ax.imshow(
        sim_sub,
        aspect="auto",
        interpolation="nearest",
        cmap=cmap10,
        norm=norm,
    )
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, boundaries=bounds)

    # No title, and no matching markers.
    ax.set_xlabel(db2)
    ax.set_ylabel(db1)

    xt, xl = tv._make_tick_positions_labels(
        info2,
        cols,
        label_mode="idx",
        max_tick_labels=int(max_tick_labels),
        force_all=False,
    )
    yt, yl = tv._make_tick_positions_labels(
        info1,
        rows,
        label_mode="idx",
        max_tick_labels=int(max_tick_labels),
        force_all=False,
    )
    ax.set_xticks(xt)
    ax.set_xticklabels(xl, rotation=90, fontsize=8)
    ax.set_yticks(yt)
    ax.set_yticklabels(yl, fontsize=8)
    fig.tight_layout()

    out_name = (
        f"sim_only_{_safe_name(seq)}_{_safe_name(ion)}_"
        f"{_safe_name(db1)}_vs_{_safe_name(db2)}_"
        f"db1idx_{_format_range_for_name(db1_idx_min, db1_idx_max)}_"
        f"db2idx_{_format_range_for_name(db2_idx_min, db2_idx_max)}.pdf"
    )
    out_path = PHOTO_DIR / out_name
    fig.savefig(out_path, bbox_inches="tight")

    print(f"[SAVE] {out_path}")
    print(f"matrix shape = {sim.shape}, selected shape = {sim_sub.shape}")
    if sim_summary:
        print("similarity summary =", sim_summary)

    display(fig)
    plt.close(fig)
    return out_path


def _on_draw_clicked(_):
    with out:
        clear_output(wait=True)
        try:
            plot_similarity_only_pdf(
                seq=seq_w.value,
                ion=ion_w.value,
                db1=db1_w.value,
                db2=db2_w.value,
                db1_idx_min=_parse_optional_int(db1_min_w.value),
                db1_idx_max=_parse_optional_int(db1_max_w.value),
                db2_idx_min=_parse_optional_int(db2_min_w.value),
                db2_idx_max=_parse_optional_int(db2_max_w.value),
                figsize=(float(fig_w_w.value), float(fig_h_w.value)),
                max_tick_labels=int(max_tick_w.value),
            )
        except Exception as e:
            print("[ERROR]", repr(e))
            raise


draw_btn.on_click(_on_draw_clicked)
print("Ready. Choose parameters above, then click: Draw and save PDF")
